<div align="center">

# Week 13: Digital Twins II: What-If Analysis and Predictive Maintenance

**Cyber-Physical Systems (BIL-435), hands-on Colab notebook**

**Prof. Dr. Utku Kose**

Full Professor, Department of Computer Engineering, Süleyman Demirel University, Isparta, Türkiye  
Founding Director, AI Application and Research Center (YAZEM), Süleyman Demirel University  
Head of the Computer Science Division, Department of Computer Engineering, Süleyman Demirel University  
Additional affiliations: University of North Dakota (USA), Universidad Panamericana (Mexico City, Mexico), Vel Tech University (Chennai, India)  
IEEE Senior Member, ACM Professional Member

[ORCID 0000-0002-9652-6415](https://orcid.org/0000-0002-9652-6415) | [utkukose.com](https://www.utkukose.com) | [github.com/utkukose](https://github.com/utkukose)

[utkukose@sdu.edu.tr](mailto:utkukose@sdu.edu.tr) | [utku.kose@und.edu](mailto:utku.kose@und.edu) | [ukose@up.edu.mx](mailto:ukose@up.edu.mx) | [utkukose@gmail.com](mailto:utkukose@gmail.com)

[Week overview](https://github.com/utkukose/cyber-physical-systems-NB-lecture/tree/main/weeks/week-13) · [Lecture page](https://utkukose.github.io/cyber-physical-systems-NB-lecture/weeks/week-13/lecture.html) · [Interactive lab](https://utkukose.github.io/cyber-physical-systems-NB-lecture/weeks/week-13/lab.html) · [PDF notes](https://utkukose.github.io/cyber-physical-systems-NB-lecture/weeks/week-13/Week13_Lecture_Notes.pdf)

</div>

## How to use this Colab notebook

This Colab notebook is the hands-on part of the week. The concepts are explained on the [lecture page](https://utkukose.github.io/cyber-physical-systems-NB-lecture/weeks/week-13/lecture.html), and each section that applies a part of the lecture starts with a link to it. Save your own copy first with File, Save a copy in Drive, then run the cells in order with Shift+Enter. Exercises check your answer as soon as you run them. Each exercise has a reference solution in a collapsed cell: Try first, then open it. Running all cells at once also runs the reference solutions. Cells titled *Observe and reflect* ask open questions about what the code shows. They have no solution cell, and a short answer in a text cell of your copy (Insert, Text cell) is enough. Cells titled *Python note* explain a piece of the Python language used nearby. Nothing needs to be installed.

**Starting in the middle.** Each numbered section can also be opened on its own, for example from an orange Colab box of the lecture page. Its first code cell, *Catch-up*, runs the code of the earlier sections without their printouts and figures. When the Colab notebook runs from the top, the catch-up cells notice it and do nothing.

## 0. Setup

The first cell prepares the environment and defines the helper `check`, which compares an answer with the expected value and prints immediate feedback. It also defines `lab_rng`, the random generator of the interactive lab, and builds the history of the library of Week 12 with its observed week `WEEK`. The calibrated twin of Week 12 is kept in `TWIN`, and the capacity tests of the battery in `BAT`. The second cell keeps the expected values of the exercises in the table `ANSWERS`, with its code hidden, so that no answer stands next to its exercise.

In [ ]:
# Environment: Everything used by this course is preinstalled in Google Colab.
try:
    import google.colab  # noqa: F401
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

import io
import math
import warnings
warnings.filterwarnings("ignore")
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates

plt.rcParams.update({"figure.dpi": 110, "figure.figsize": (7, 3.6),
                     "axes.grid": True, "grid.alpha": 0.3})

def check(name, value, expected=None, tol=1e-6):
    """Compares an answer with the expected value and prints immediate feedback.
    Without an expected value in the call, check takes the value and its tolerance
    from ANSWERS, the table of the next cell."""
    if expected is None:
        answer, tol = ANSWERS[name]
        expected = answer()
    if value is None:
        print(f"{name}: not answered yet. Replace None and run the cell again.")
        return False
    try:
        scale = max(1.0, abs(float(expected)))
        ok = abs(float(value) - float(expected)) <= tol * scale
    except (TypeError, ValueError):
        ok = value == expected
    print(f"{name}: " + ("correct." if ok else
                         "not yet. Compare your steps with the lecture."))
    return ok

def lab_rng(seed):
    """The random generator of the interactive lab, so that the Colab notebook draws exactly the same numbers."""
    s = seed & 0xFFFFFFFF
    def uniform():
        nonlocal s
        s = (s + 0x6D2B79F5) & 0xFFFFFFFF
        t = ((s ^ (s >> 15)) * (s | 1)) & 0xFFFFFFFF
        t = (t ^ ((t + (((t ^ (t >> 7)) * (t | 61)) & 0xFFFFFFFF)) & 0xFFFFFFFF)) & 0xFFFFFFFF
        return ((t ^ (t >> 14)) & 0xFFFFFFFF) / 4294967296
    def normal():
        u = v = 0.0
        while u == 0.0:
            u = uniform()
        while v == 0.0:
            v = uniform()
        return math.sqrt(-2 * math.log(u)) * math.cos(2 * math.pi * v)
    return uniform, normal

# The library B2 of the campus as it really behaves: the air and the furniture, the mass of the walls, the envelope and
# the gaps of the windows (a synthetic history, built from the model of Week 2 with a second heat capacity)
C_AIR, C_MASS = 20.0, 10.0          # kWh/K
R_AIR_MASS, R_MASS_OUT = 0.02, 0.46  # K/kW: from the air into the walls, from the walls to the outside
R_GAPS, R_WINDOW = 6.0, 0.7          # K/kW: the gaps of the closed windows, one window wide open
OPEN_FROM, OPEN_TO = pd.Timestamp("2026-02-18 10:00"), pd.Timestamp("2026-02-19 16:00")

def library_history(seed=12):
    """Three weeks of the library from Monday 2 February 2026, simulated minute by minute and logged every 15 minutes:
    the indoor sensor (noise 0.1 K), the outdoor temperature, the mean heater power, the sun on the windows and the
    opening hours. The thermostat holds 21 C from 6:00 (8:00 at weekends) to closing time and 17 C at night."""
    rng = np.random.default_rng(seed)
    days, n = 21, 21 * 24 * 60
    t = np.arange(n) / 60.0
    hour, day = t % 24, (t // 24).astype(int)
    weekday = day % 7
    level = np.zeros(days)
    for d in range(1, days):
        level[d] = 0.6 * level[d - 1] + rng.normal(0, 2.0)          # the weather of each day
    t_out = 3 + 4 * np.sin(2 * np.pi * (hour - 9) / 24) + np.interp(t, np.arange(days) * 24 + 12, level)
    cloud = rng.uniform(0.25, 1.0, days)
    sun = 450 * np.maximum(0, np.sin(np.pi * (hour - 7.5) / 9.5)) * cloud[day]
    is_open = np.where(weekday < 5, (hour >= 8) & (hour < 22), (hour >= 10) & (hour < 18))
    heated = np.where(weekday < 5, (hour >= 6) & (hour < 22), (hour >= 8) & (hour < 18))
    visitors = rng.uniform(0.6, 1.4, days)                          # people, lights and computers: about 6 kW
    gains = 6.0 * is_open * visitors[day]
    r_gaps = np.full(n, R_GAPS)
    r_gaps[(t >= (OPEN_FROM - pd.Timestamp("2026-02-02")) / pd.Timedelta(hours=1))
           & (t < (OPEN_TO - pd.Timestamp("2026-02-02")) / pd.Timedelta(hours=1))] = R_WINDOW
    air, mass, heat = np.zeros(n), np.zeros(n), np.zeros(n)
    air[0] = mass[0] = 18.0
    on = False
    for k in range(n - 1):
        setpoint = 21.0 if heated[k] else 17.0
        if air[k] < setpoint - 0.5:
            on = True
        elif air[k] > setpoint + 0.5:
            on = False
        heat[k] = 60.0 if on else 0.0
        q_air = (mass[k] - air[k]) / R_AIR_MASS + (t_out[k] - air[k]) / r_gaps[k] + heat[k] + 0.01 * sun[k] + gains[k]
        q_mass = (air[k] - mass[k]) / R_AIR_MASS + (t_out[k] - mass[k]) / R_MASS_OUT + 0.01 * sun[k]
        air[k + 1] = air[k] + q_air / C_AIR / 60
        mass[k + 1] = mass[k] + q_mass / C_MASS / 60
    heat[-1] = heat[-2]
    mean15 = lambda a: a.reshape(-1, 15).mean(axis=1)
    index = pd.date_range("2026-02-02", periods=n // 15, freq="15min")
    true = air[::15]
    return pd.DataFrame({"t_true": true.round(3), "t_in": (true + rng.normal(0, 0.1, len(true))).round(2),
                         "t_out": mean15(t_out).round(2), "heater_kw": mean15(heat).round(2),
                         "solar_wm2": mean15(sun).round(1), "open": mean15(is_open.astype(float))}, index=index)

HIST = library_history()
WEEK = HIST.loc["2026-02-09":"2026-02-15"]       # the observed week of the what-if analysis, Monday to Sunday
TWIN = dict(C=30.41, R=0.440, g=0.0178, q=7.33)  # the twin of the library calibrated in Week 12
DT = 0.25                                         # hours between two rows

# The battery of the campus (500 kWh) since its commissioning on 1 November 2023: a capacity test at the end of every
# month and the equivalent full cycles (EFC) counted by the battery management system (a synthetic history)
Q0, A_TRUE, B_TRUE = 500.0, 0.025, 0.00004       # kWh; calendar ageing per square root of a year; ageing per cycle

def battery_history(seed=13, months=36):
    """Monthly capacity tests of the battery, with a noise of 1.5 kWh, rounded as the battery management system logs
    them. More cycles in spring and summer, when the battery stores the midday sun."""
    uniform, normal = lab_rng(seed)
    rows, cycles = [], 0.0
    for m in range(months):
        per_day = (1.0 + 0.2 * math.sin(2 * math.pi * (m - 3) / 12)) * (0.92 + 0.16 * uniform())
        cycles += per_day * 365.25 / 12
        years = (m + 1) / 12
        health = 1 - A_TRUE * math.sqrt(years) - B_TRUE * cycles
        rows.append((pd.Timestamp("2023-11-01") + pd.DateOffset(months=m + 1), years, round(cycles, 1),
                     round(Q0 * health + 1.5 * normal(), 1)))
    return pd.DataFrame(rows, columns=["test_date", "years", "efc", "capacity_kwh"])

BAT = battery_history()
NOW = 3.0                                          # years since the commissioning: 1 November 2026
print("Setup complete. Running in Colab:", IN_COLAB, "|", len(WEEK), "rows of the library and", len(BAT),
      "capacity tests of the battery")

In [ ]:
#@title Expected answers of the exercises (run this cell; open it only after trying the exercises)
ANSWERS = {
    "Exercise 13.1": (lambda: max(scenario_kwh.values()), 1e-6),
    "Exercise 13.2": (lambda: np.mean(np.random.default_rng(13).normal(3.0, 2.5, 20000) < 0.0), 1e-9),
    "Exercise 13.3": (lambda: np.percentile(np.random.default_rng(13).exponential(5.0, 1000), 10), 1e-9),
    "Exercise A": (lambda: schedule_twin(WEEK, night=15, optimal=True, temp0=T0)['energy'], 1e-6),
    "Exercise B": (lambda: monte_carlo(13, 'optimal')['p_deficit'], 1e-9),
    "Exercise C": (lambda: round(0.2 / fit_linear(BAT), 2), 1e-9),
    "Exercise D": (lambda: latest_order(1.25)[0], 1e-9),
    "Exercise E": (lambda: STAGE13['frequency regulation: latest order (months from now)'], 1e-9),
}

## Python step 13: Scenarios and Monte Carlo

A what-if question runs the same model under several assumptions, and a Monte Carlo study runs it under many random ones. This step loops over named scenarios, draws random numbers with NumPy and turns many random runs into a probability &#91;[2](https://utkukose.github.io/cyber-physical-systems-NB-lecture/weeks/week-13/lecture.html#ref-2), [3](https://utkukose.github.io/cyber-physical-systems-NB-lecture/weeks/week-13/lecture.html#ref-3)&#93;.

**Started here?** If you opened the Colab notebook at this section, run the next cell once. It runs the code of the earlier sections for you, without their printouts and figures. This takes from a few seconds to a minute, and the cell does nothing when they have already run. The Colab menu Runtime, Run before, which runs every cell above the selected one, works as well.

In [ ]:
#@title Catch-up: run this cell once if you start the notebook here
if "ANSWERS" in globals() and "BAT" in globals():
    print("The earlier sections have already run. Continue with the next cell.")
else:
    import json, urllib.request
    import matplotlib.pyplot as plt
    from IPython import get_ipython
    from IPython.utils.capture import capture_output
    _cu_url = ("https://raw.githubusercontent.com/utkukose/"
               "cyber-physical-systems-NB-lecture/main/weeks/week-13/"
               "NB13_digital_twin_whatif_maintenance.ipynb")
    _cu_cells = json.load(urllib.request.urlopen(_cu_url))["cells"]
    _cu_ip, _cu_err = get_ipython(), None
    with capture_output():
        for _cu_c in _cu_cells:
            if _cu_c.get("id") == "python-step":
                break
            _cu_meta = _cu_c.get("metadata", {})
            if _cu_c["cell_type"] == "code" and _cu_meta.get("catchup", True):
                try:
                    _cu_src = _cu_ip.transform_cell("".join(_cu_c["source"]))
                    exec(_cu_src, _cu_ip.user_ns)
                except Exception as _cu_e:
                    _cu_err = _cu_e
                    break
    plt.close("all")
    if _cu_err is None:
        print("The earlier sections have run. Continue with the next cell.")
    else:
        print(f"The catch-up stopped with an error ({_cu_err}). "
              "Run the notebook from the top instead.")

### A loop over scenarios

In [ ]:
# Python practice: a small example for learning the language
scenarios = {"mild": 8.0, "normal": 3.0, "cold": -4.0}     # outdoor temperature of a day (C)
scenario_kwh = {}
for name, t_out in scenarios.items():
    scenario_kwh[name] = round((21.0 - t_out) / 0.44 * 24, 1)    # heat that holds 21 C for 24 h (Week 2)
print(scenario_kwh)

A dictionary gives each scenario a name, and the method `items` returns the pairs of name and value. The loop runs the same model for every scenario and stores the results under the same names.

### Random numbers with NumPy

In [ ]:
# Python practice: a small example for learning the language
rng = np.random.default_rng(13)                 # a generator with a seed: the same numbers in every run
t_out = rng.normal(3.0, 2.5, size=10000)         # 10000 possible daily means of the outdoor temperature
print("mean", t_out.mean().round(2), "| spread", t_out.std().round(2))
print("5, 50 and 95 percent quantiles:", np.percentile(t_out, [5, 50, 95]).round(2))

The function `default_rng` creates a random generator. The seed makes the numbers repeatable, which matters for checking results. The method `normal` draws from a normal distribution with a mean and a spread, and `np.percentile` returns the values below which a given percentage of the draws lies.

### A probability from many random runs

In [ ]:
# Python practice: a small example for learning the language
heat_kwh = (21.0 - t_out) / 0.44 * 24            # the model applied to all 10000 days at once
print("days that need more than 1100 kWh:", np.mean(heat_kwh > 1100).round(4))
lamps = rng.exponential(5.0, size=(1000, 20))   # 1000 possible years of 20 lamps, mean life 5 years
failures = (lamps < 1.0).sum(axis=1)            # lamps that fail within the first year
print("failed lamps per year: mean", failures.mean(), "| at most 6 in", np.mean(failures <= 6) * 100, "percent of years")

A comparison like `heat_kwh > 1100` gives an array of True and False, and its mean is the share of True. This share estimates a probability, the core of the Monte Carlo method. The method `exponential` draws lifetimes with a constant failure rate, and `sum(axis=1)` counts along each row.

**Quick check.** Why does the estimate of the probability change a little when the seed changes, and how can it be made more precise?

<details><summary>Answer</summary>

Every seed gives other random numbers, so the share of days above 1100 kWh varies around the true probability. More draws make the estimate more precise: Its error shrinks with the square root of the number of draws.

</details>

### Exercises for Python step 13

**Exercise 13.1.** Store the largest daily heat of the three scenarios above in `kwh_131`.

In [ ]:
kwh_131 = None   # your answer

check("Exercise 13.1", kwh_131)

In [ ]:
#@title Reference solution (run this cell; open it only after trying)
kwh_131 = max(scenario_kwh.values())

check("Exercise 13.1", kwh_131)

**Exercise 13.2.** Draw 20000 outdoor temperatures with `np.random.default_rng(13).normal(3.0, 2.5, 20000)` and store the share of days below 0 C in `p_132`.

In [ ]:
p_132 = None   # your answer

check("Exercise 13.2", p_132)

In [ ]:
#@title Reference solution (run this cell; open it only after trying)
p_132 = np.mean(np.random.default_rng(13).normal(3.0, 2.5, 20000) < 0.0)

check("Exercise 13.2", p_132)

**Exercise 13.3.** Draw 1000 lamp lifetimes with `np.random.default_rng(13).exponential(5.0, 1000)` and store the 10 percent quantile in years, the time by which one lamp in ten has failed, in `life_133`.

In [ ]:
life_133 = None   # your answer

check("Exercise 13.3", life_133)

In [ ]:
#@title Reference solution (run this cell; open it only after trying)
life_133 = np.percentile(np.random.default_rng(13).exponential(5.0, 1000), 10)

check("Exercise 13.3", life_133)

## 1. What-if: The heating schedule of the library &#91;[1](https://utkukose.github.io/cyber-physical-systems-NB-lecture/weeks/week-13/lecture.html#ref-1)&#93;

*Lecture: [What-if analysis with the twin](https://utkukose.github.io/cyber-physical-systems-NB-lecture/weeks/week-13/lecture.html#what-if)*

**Started here?** If you opened the Colab notebook at this section, run the next cell once. It runs the code of the earlier sections for you, without their printouts and figures. This takes from a few seconds to a minute, and the cell does nothing when they have already run. The Colab menu Runtime, Run before, which runs every cell above the selected one, works as well.

In [ ]:
#@title Catch-up: run this cell once if you start the notebook here
if "ANSWERS" in globals() and "scenario_kwh" in globals():
    print("The earlier sections have already run. Continue with the next cell.")
else:
    import json, urllib.request
    import matplotlib.pyplot as plt
    from IPython import get_ipython
    from IPython.utils.capture import capture_output
    _cu_url = ("https://raw.githubusercontent.com/utkukose/"
               "cyber-physical-systems-NB-lecture/main/weeks/week-13/"
               "NB13_digital_twin_whatif_maintenance.ipynb")
    _cu_cells = json.load(urllib.request.urlopen(_cu_url))["cells"]
    _cu_ip, _cu_err = get_ipython(), None
    with capture_output():
        for _cu_c in _cu_cells:
            if _cu_c.get("id") == "sec-1":
                break
            _cu_meta = _cu_c.get("metadata", {})
            if _cu_c["cell_type"] == "code" and _cu_meta.get("catchup", True):
                try:
                    _cu_src = _cu_ip.transform_cell("".join(_cu_c["source"]))
                    exec(_cu_src, _cu_ip.user_ns)
                except Exception as _cu_e:
                    _cu_err = _cu_e
                    break
    plt.close("all")
    if _cu_err is None:
        print("The earlier sections have run. Continue with the next cell.")
    else:
        print(f"The catch-up stopped with an error ({_cu_err}). "
              "Run the notebook from the top instead.")

**Why Python here.** The library is too cold when it opens. Its thermostat holds 17 C at night and starts the heating at 6:00, which is too late on cold mornings. Python puts the twin of Week 12 into a closed loop with the thermostat. It checks the twin against the observed week and then tries other schedules.

**From the formulas to the code.** The twin steps with $T_{k+1} = T_k + \Delta t/C\,((T_{out} - T_k)/R + P_k + g\,G + q\,o)$, and the thermostat switches $P_k$ between 0 and 60 kW around its setpoint with a band of $\pm 0.5$ K. The comfort deficit sums $\max(0, 20 - T_k)\,\Delta t$ over the open hours, in K h. The optimal start heats as soon as the time to reach 20.5 C with full power, $\tau \ln((T_{ss} - T)/(T_{ss} - 20.5))$ with $T_{ss} = T_{out} + 60\,R$, exceeds the time left before opening minus one step.

In [ ]:
def schedule_twin(d, night=17.0, start=6.0, optimal=False, temp0=None, visitors=None):
    """The twin in closed loop with the thermostat over the rows of d, from a Monday 00:00. The heating runs from start
    (2 hours later at weekends) until closing; optimal=True starts it as late as the twin allows instead."""
    C, R, g, q = TWIN["C"], TWIN["R"], TWIN["g"], TWIN["q"]
    t_out, sun, is_open = (np.asarray(d[c], float) for c in ("t_out", "solar_wm2", "open"))
    n = len(t_out)
    temp, heat = np.zeros(n), np.zeros(n)
    temp[0] = 18.0 if temp0 is None else temp0
    on, started = False, -1
    for k in range(n - 1):
        day, hour = k // 96, (k % 96) / 4
        opens, closes = (8, 22) if day % 7 < 5 else (10, 18)
        if optimal:
            if hour < opens and started != day:
                steady = t_out[k] + R * 60.0
                need = math.inf if steady <= 20.5 else (R * C * math.log((steady - temp[k]) / (steady - 20.5))
                                                        if temp[k] < 20.5 else 0.0)
                if need >= (opens - hour) - DT:
                    started = day
            setpoint = 21.0 if (started == day or opens <= hour) and hour < closes else night
        else:
            begin = start if day % 7 < 5 else start + 2
            setpoint = 21.0 if begin <= hour < closes else night
        if temp[k] < setpoint - 0.5:
            on = True
        elif temp[k] > setpoint + 0.5:
            on = False
        heat[k] = 60.0 if on else 0.0
        gains = q * is_open[k] * (1.0 if visitors is None else visitors[day])
        temp[k + 1] = temp[k] + DT / C * ((t_out[k] - temp[k]) / R + heat[k] + g * sun[k] + gains)
    heat[-1] = heat[-2]
    return dict(temp=temp, heat=heat, energy=round(float(heat.sum() * DT), 1),
                deficit=round(float((np.maximum(0, 20.0 - temp) * is_open).sum() * DT), 3),
                hours_below=float(((temp < 20.0) & (is_open > 0)).sum() * DT))

T0 = WEEK.t_in.iloc[0]                                    # the measured temperature on Monday 00:00
now = schedule_twin(WEEK, temp0=T0)
real_deficit = float((np.maximum(0, 20.0 - WEEK.t_true) * WEEK.open).sum() * DT)
print(f"current schedule, twin: {now['energy']} kWh, comfort deficit {now['deficit']} K h, {now['hours_below']} h below 20 C")
print(f"current schedule, library: {WEEK.heater_kw.sum() * DT:.1f} kWh, comfort deficit {real_deficit:.3f} K h")
grid = {}
for night in (11, 13, 15, 17, 19):
    for start in (4, 5, 6, 7, "optimal"):
        run = schedule_twin(WEEK, night=night, start=0 if start == "optimal" else start, optimal=start == "optimal",
                            temp0=T0)
        grid[(night, start)] = (run["energy"], run["deficit"])
table = pd.DataFrame({s: [f"{grid[(n, s)][0]:.0f} kWh, {grid[(n, s)][1]:.2f} K h" for n in (11, 13, 15, 17, 19)]
                      for s in (4, 5, 6, 7, "optimal")}, index=[f"night {n} C" for n in (11, 13, 15, 17, 19)])
print(table.to_string())

**Python note.** The function `schedule_twin` returns a dictionary, so `now['energy']` reads one result. The key `(night, start)` of the dictionary `grid` is a tuple, which can hold a number and a word together.

**Exercise A.** Store the heating energy in kWh of the observed week with 15 C at night and the optimal start in `kwh_a`.

In [ ]:
kwh_a = None   # your answer

check("Exercise A", kwh_a)

In [ ]:
#@title Reference solution (run this cell; open it only after trying)
kwh_a = schedule_twin(WEEK, night=15, optimal=True, temp0=T0)['energy']

check("Exercise A", kwh_a)

In [ ]:
# Figure: the observed week with the current schedule and with 13 C at night and the optimal start
best = schedule_twin(WEEK, night=13, optimal=True, temp0=T0)
fig, ax = plt.subplots(2, 1, figsize=(10, 4.8), sharex=True, gridspec_kw={"height_ratios": [2, 1]})
ax[0].fill_between(WEEK.index, 15, 23, where=WEEK.open > 0, color="#2E8B57", alpha=0.08, step="post", label="open")
ax[0].plot(WEEK.index, now["temp"], color="#C98510", lw=1, label="17 C at night, start 6:00")
ax[0].plot(WEEK.index, best["temp"], color="#1E6FA8", lw=1, label="13 C at night, optimal start")
ax[0].axhline(20, color="#B53F2E", ls=":", lw=1)
ax[0].set_ylim(12.5, 22.5)
ax[0].set_ylabel("twin temperature (C)")
ax[0].legend(fontsize=8, loc="lower center", ncol=3)
ax[1].plot(WEEK.index, np.cumsum(now["heat"]) * DT, color="#C98510", label="17 C, 6:00")
ax[1].plot(WEEK.index, np.cumsum(best["heat"]) * DT, color="#1E6FA8", label="13 C, optimal start")
ax[1].set_ylabel("heat since Monday (kWh)")
ax[1].legend(fontsize=8, loc="upper left")
ax[1].xaxis.set_major_formatter(mdates.DateFormatter("%a %d"))
plt.tight_layout()
plt.show()

**Observe and reflect.** Why does the optimal start with 13 C at night need hardly more energy than the current schedule, and yet keep the library warm at opening?

## 2. Monte Carlo: Many possible weeks &#91;[3](https://utkukose.github.io/cyber-physical-systems-NB-lecture/weeks/week-13/lecture.html#ref-3)&#93;

*Lecture: [Uncertainty: Many possible weeks](https://utkukose.github.io/cyber-physical-systems-NB-lecture/weeks/week-13/lecture.html#monte-carlo)*

**Started here?** If you opened the Colab notebook at this section, run the next cell once. It runs the code of the earlier sections for you, without their printouts and figures. This takes from a few seconds to a minute, and the cell does nothing when they have already run. The Colab menu Runtime, Run before, which runs every cell above the selected one, works as well.

In [ ]:
#@title Catch-up: run this cell once if you start the notebook here
if "ANSWERS" in globals() and "schedule_twin" in globals():
    print("The earlier sections have already run. Continue with the next cell.")
else:
    import json, urllib.request
    import matplotlib.pyplot as plt
    from IPython import get_ipython
    from IPython.utils.capture import capture_output
    _cu_url = ("https://raw.githubusercontent.com/utkukose/"
               "cyber-physical-systems-NB-lecture/main/weeks/week-13/"
               "NB13_digital_twin_whatif_maintenance.ipynb")
    _cu_cells = json.load(urllib.request.urlopen(_cu_url))["cells"]
    _cu_ip, _cu_err = get_ipython(), None
    with capture_output():
        for _cu_c in _cu_cells:
            if _cu_c.get("id") == "sec-2":
                break
            _cu_meta = _cu_c.get("metadata", {})
            if _cu_c["cell_type"] == "code" and _cu_meta.get("catchup", True):
                try:
                    _cu_src = _cu_ip.transform_cell("".join(_cu_c["source"]))
                    exec(_cu_src, _cu_ip.user_ns)
                except Exception as _cu_e:
                    _cu_err = _cu_e
                    break
    plt.close("all")
    if _cu_err is None:
        print("The earlier sections have run. Continue with the next cell.")
    else:
        print(f"The catch-up stopped with an error ({_cu_err}). "
              "Run the notebook from the top instead.")

**Why Python here.** The observed week is only one week. Next week may be colder, sunnier or busier. Python draws 300 possible weeks of weather and visitors with the random generator of the interactive lab and runs every schedule through all of them.

**From the formulas to the code.** Each week draws the daily level of the outdoor temperature as $\ell_d = 0.6\,\ell_{d-1} + 2\,\varepsilon_d$, with $\ell_0 = 2.5\,\varepsilon_0$ and standard normal $\varepsilon$. It also draws a cloud factor between 0.25 and 1 and a visitor factor between 0.6 and 1.4: `weather(seed)`. The probability of a comfort deficit above 1 K h is the share of such weeks: `monte_carlo(night, start)`.

In [ ]:
def weather(seed):
    """One possible week of weather and visitors, drawn in the same order as in the interactive lab."""
    uniform, normal = lab_rng(seed)
    level, cloud, visitors = np.zeros(7), np.zeros(7), np.zeros(7)
    for d in range(7):
        level[d] = 2.5 * normal() if d == 0 else 0.6 * level[d - 1] + 2.0 * normal()
        cloud[d] = 0.25 + 0.75 * uniform()
        visitors[d] = 0.6 + 0.8 * uniform()
    hours = (np.arange(672) + 0.5) / 4                      # the middle of each row, hours since Monday 00:00
    hour, day = hours % 24, (hours // 24).astype(int)
    t_out = 3 + 4 * np.sin(2 * np.pi * (hour - 9) / 24) + np.interp(hours, 12 + 24 * np.arange(7), level)
    sun = 450 * np.maximum(0, np.sin(np.pi * (hour - 7.5) / 9.5)) * cloud[day]
    is_open = np.where(day % 7 < 5, (hour >= 8) & (hour < 22), (hour >= 10) & (hour < 18)).astype(float)
    return dict(t_out=t_out, solar_wm2=sun, open=is_open), visitors

def monte_carlo(night, start, weeks=300, first_seed=1300):
    """The schedule over many possible weeks: the energy and the comfort deficit of each week, and their summary."""
    energy, deficit = np.zeros(weeks), np.zeros(weeks)
    for i in range(weeks):
        d, visitors = weather(first_seed + i)
        run = schedule_twin(d, night=night, start=0 if start == "optimal" else start, optimal=start == "optimal",
                            visitors=visitors)
        energy[i], deficit[i] = run["energy"], run["deficit"]
    return dict(energy=energy, deficit=deficit, mean_kwh=round(float(energy.mean()), 1),
                p90_kwh=round(float(np.percentile(energy, 90)), 1), mean_deficit=round(float(deficit.mean()), 3),
                p_deficit=round(float(np.mean(deficit > 1.0)), 3))

MC = {}
for night, start in ((17, 6), (19, 6), (17, "optimal"), (15, "optimal"), (13, "optimal")):
    MC[(night, start)] = monte_carlo(night, start)
    m = MC[(night, start)]
    print(f"night {night} C, start {start}: energy mean {m['mean_kwh']} kWh, 90 percent quantile {m['p90_kwh']} kWh | "
          f"deficit mean {m['mean_deficit']} K h, P(deficit > 1 K h) = {m['p_deficit']}")

**Python note.** Each week uses its own seed, `first_seed + i`, so the 300 weeks differ and every run repeats them exactly. The interactive lab draws the same weeks with the same seeds.

**Exercise B.** Store the probability of a comfort deficit above 1 K h with 13 C at night and the optimal start in `p_b`.

In [ ]:
p_b = None   # your answer

check("Exercise B", p_b)

In [ ]:
#@title Reference solution (run this cell; open it only after trying)
p_b = monte_carlo(13, 'optimal')['p_deficit']

check("Exercise B", p_b)

Part A of the [interactive lab](https://utkukose.github.io/cyber-physical-systems-NB-lecture/weeks/week-13/lab.html) opens the plant room of the library with the controller of the heating. Paste the copied rows into `MY_SCHEDULES`, and the code repeats each of your runs.

In [ ]:
# Paste the results of lab Part A between the triple quotes.
MY_SCHEDULES = """
"""
if MY_SCHEDULES.strip():
    runs = pd.read_csv(io.StringIO(MY_SCHEDULES.strip()))
    source = "your own schedules in the plant room"
else:
    runs = pd.DataFrame({"run": [1, 2, 3], "night_c": [17, 13, 13], "start": ["6", "optimal", "optimal"],
                         "weeks": [1, 1, 300], "energy_kwh": [5730.0, 5760.0, 5420.4], "deficit_kh": [8.791, 0.0, 0.222]})
    source = "a simulated student"
def repeat(r):
    start = "optimal" if str(r.start) == "optimal" else float(r.start)
    if r.weeks == 1:
        run = schedule_twin(WEEK, night=r.night_c, start=0 if start == "optimal" else start,
                            optimal=start == "optimal", temp0=T0)
        return run["energy"], run["deficit"]
    m = monte_carlo(r.night_c, start, weeks=int(r.weeks))
    return m["mean_kwh"], m["mean_deficit"]
runs[["model_energy_kwh", "model_deficit_kh"]] = [repeat(r) for r in runs.itertuples()]
print("Data:", source)
print(runs.to_string(index=False))

In [ ]:
# Figure: the energy and the comfort deficit of 300 possible weeks for three schedules
fig, ax = plt.subplots(1, 2, figsize=(11, 3.4))
for key, col, name in (((17, 6), "#C98510", "17 C, 6:00"), ((19, 6), "#56616D", "19 C, 6:00"),
                       ((13, "optimal"), "#1E6FA8", "13 C, optimal start")):
    ax[0].hist(MC[key]["energy"], bins=30, color=col, alpha=0.45, label=name)
    ax[1].hist(np.minimum(MC[key]["deficit"], 25), bins=50, color=col, alpha=0.45, label=name)
ax[0].set_xlabel("heating energy of the week (kWh)")
ax[1].set_xlabel("comfort deficit of the week (K h, cut at 25)")
ax[1].axvline(1.0, color="#B53F2E", ls=":", lw=1)
for a in ax:
    a.set_ylabel("weeks")
    a.legend(fontsize=8)
plt.tight_layout()
plt.show()

**Observe and reflect.** Which schedule would you recommend for the library, and which numbers of the 300 weeks support it?

## 3. Predictive maintenance: The capacity of the battery &#91;[5](https://utkukose.github.io/cyber-physical-systems-NB-lecture/weeks/week-13/lecture.html#ref-5)&#93;

*Lecture: [Predictive maintenance and the remaining useful life](https://utkukose.github.io/cyber-physical-systems-NB-lecture/weeks/week-13/lecture.html#remaining-useful-life)*

**Started here?** If you opened the Colab notebook at this section, run the next cell once. It runs the code of the earlier sections for you, without their printouts and figures. This takes from a few seconds to a minute, and the cell does nothing when they have already run. The Colab menu Runtime, Run before, which runs every cell above the selected one, works as well.

In [ ]:
#@title Catch-up: run this cell once if you start the notebook here
if "ANSWERS" in globals() and "monte_carlo" in globals():
    print("The earlier sections have already run. Continue with the next cell.")
else:
    import json, urllib.request
    import matplotlib.pyplot as plt
    from IPython import get_ipython
    from IPython.utils.capture import capture_output
    _cu_url = ("https://raw.githubusercontent.com/utkukose/"
               "cyber-physical-systems-NB-lecture/main/weeks/week-13/"
               "NB13_digital_twin_whatif_maintenance.ipynb")
    _cu_cells = json.load(urllib.request.urlopen(_cu_url))["cells"]
    _cu_ip, _cu_err = get_ipython(), None
    with capture_output():
        for _cu_c in _cu_cells:
            if _cu_c.get("id") == "sec-3":
                break
            _cu_meta = _cu_c.get("metadata", {})
            if _cu_c["cell_type"] == "code" and _cu_meta.get("catchup", True):
                try:
                    _cu_src = _cu_ip.transform_cell("".join(_cu_c["source"]))
                    exec(_cu_src, _cu_ip.user_ns)
                except Exception as _cu_e:
                    _cu_err = _cu_e
                    break
    plt.close("all")
    if _cu_err is None:
        print("The earlier sections have run. Continue with the next cell.")
    else:
        print(f"The catch-up stopped with an error ({_cu_err}). "
              "Run the notebook from the top instead.")

**Why Python here.** The battery of the campus loses capacity with time and with use. The battery management system tests the capacity at the end of every month. Python fits two models of ageing to three years of tests and predicts when the capacity falls to 80 percent, 400 kWh, the end of life of the battery.

**From the formulas to the code.** The linear model is $1 - Q/Q_0 = c\,t$: `fit_linear`. The model of calendar and cycle ageing is $1 - Q/Q_0 = a\sqrt{t} + b\,N$, with the age $t$ in years and the equivalent full cycles $N$: `fit_ageing`. Both are linear in their parameters, so least squares solves them directly. With $n$ cycles per day from now on, $N(t) = N_{now} + 365.25\,n\,(t - t_{now})$, and the end of life is the root of $a\sqrt{t} + b\,N(t) = 0.2$: `end_of_life`.

In [ ]:
def fit_linear(d):
    """Least squares of 1 - Q/Q0 = c t: returns c per year."""
    t, y = d.years.values, 1 - d.capacity_kwh.values / Q0
    return float(t @ y / (t @ t))

def fit_ageing(d):
    """Least squares of 1 - Q/Q0 = a sqrt(t) + b N with the normal equations; returns a, b and their covariance."""
    x1, x2, y = np.sqrt(d.years.values), d.efc.values, 1 - d.capacity_kwh.values / Q0
    s11, s12, s22 = x1 @ x1, x1 @ x2, x2 @ x2
    det = s11 * s22 - s12 ** 2
    a = (s22 * (x1 @ y) - s12 * (x2 @ y)) / det
    b = (s11 * (x2 @ y) - s12 * (x1 @ y)) / det
    resid = y - a * x1 - b * x2
    s2 = resid @ resid / (len(y) - 2)                       # variance of the residuals
    cov = s2 * np.array([[s22, -s12], [-s12, s11]]) / det
    return float(a), float(b), cov

def end_of_life(a, b, per_day, t_now=NOW, n_now=None, limit=0.2):
    """Years from the commissioning until a sqrt(t) + b N(t) reaches the limit, with per_day cycles from now on."""
    n_now = BAT.efc.iloc[-1] if n_now is None else n_now
    k = b * 365.25 * per_day
    c0 = b * (n_now - 365.25 * per_day * t_now) - limit
    x = (-a + math.sqrt(a * a - 4 * k * c0)) / (2 * k)    # the root in x = sqrt(t)
    return x * x

c = fit_linear(BAT)
A, B, COV = fit_ageing(BAT)
print(f"tests: {len(BAT)}, last {BAT.test_date.iloc[-1].date()}: {BAT.capacity_kwh.iloc[-1]} kWh after {BAT.efc.iloc[-1]} cycles")
print(f"linear model: c = {c:.5f} per year, end of life after {0.2 / c:.2f} years")
print(f"calendar and cycle model: a = {A:.5f}, b = {B:.3e} per cycle, end of life after {end_of_life(A, B, 1.0):.2f} years")
for name, fitted in (("linear", c * BAT.years), ("calendar and cycle", A * np.sqrt(BAT.years) + B * BAT.efc)):
    err = (1 - BAT.capacity_kwh / Q0 - fitted) * Q0
    print(f"  RMSE of the {name} model: {np.sqrt(np.mean(err ** 2)):.2f} kWh")
for per_day in (1.0, 1.25, 1.5):
    eol = end_of_life(A, B, per_day)
    print(f"{per_day} cycles per day from now on: end of life after {eol:.2f} years, remaining useful life {eol - NOW:.2f} years")

**Python note.** The product `x1 @ x2` of two arrays is the sum of their element-wise products. The normal equations of two parameters need only these sums, so the fit fits in six lines.

**Exercise C.** Store the end of life in years of the linear model, rounded to two decimals, in `eol_c`.

In [ ]:
eol_c = None   # your answer

check("Exercise C", eol_c)

In [ ]:
#@title Reference solution (run this cell; open it only after trying)
eol_c = round(0.2 / fit_linear(BAT), 2)

check("Exercise C", eol_c)

In [ ]:
# Figure: the capacity tests with the two models up to the end of life
years = np.linspace(0.05, 10, 300)
future = BAT.efc.iloc[-1] + 365.25 * 1.0 * (years - NOW)
efc_path = np.where(years <= NOW, np.interp(years, BAT.years, BAT.efc), future)
fig, ax = plt.subplots(figsize=(10, 3.6))
ax.plot(BAT.years, BAT.capacity_kwh, "o", ms=3, color="#1B2430", label="capacity tests")
ax.plot(years, Q0 * (1 - c * years), color="#C98510", lw=1.2, label="linear model")
ax.plot(years, Q0 * (1 - A * np.sqrt(years) - B * efc_path), color="#1E6FA8", lw=1.4,
        label="calendar and cycle model, 1 cycle per day")
ax.axhline(400, color="#B53F2E", ls=":", lw=1)
ax.axvline(NOW, color="#56616D", ls=":", lw=1)
ax.text(NOW + 0.05, 492, "now", fontsize=8)
ax.text(0.1, 403, "end of life: 400 kWh", fontsize=8, color="#B53F2E")
ax.set_xlabel("years since the commissioning on 1 November 2023")
ax.set_ylabel("capacity (kWh)")
ax.set_ylim(380, 505)
ax.legend(fontsize=8, loc="lower left")
plt.tight_layout()
plt.show()

**Observe and reflect.** Why does the linear model predict an earlier end of life, although it fits the tests of the first three years almost as well?

## 4. The uncertainty of the end of life &#91;[7](https://utkukose.github.io/cyber-physical-systems-NB-lecture/weeks/week-13/lecture.html#ref-7)&#93;

*Lecture: [The uncertainty of the remaining useful life](https://utkukose.github.io/cyber-physical-systems-NB-lecture/weeks/week-13/lecture.html#rul-uncertainty)*

**Started here?** If you opened the Colab notebook at this section, run the next cell once. It runs the code of the earlier sections for you, without their printouts and figures. This takes from a few seconds to a minute, and the cell does nothing when they have already run. The Colab menu Runtime, Run before, which runs every cell above the selected one, works as well.

In [ ]:
#@title Catch-up: run this cell once if you start the notebook here
if "ANSWERS" in globals() and "end_of_life" in globals():
    print("The earlier sections have already run. Continue with the next cell.")
else:
    import json, urllib.request
    import matplotlib.pyplot as plt
    from IPython import get_ipython
    from IPython.utils.capture import capture_output
    _cu_url = ("https://raw.githubusercontent.com/utkukose/"
               "cyber-physical-systems-NB-lecture/main/weeks/week-13/"
               "NB13_digital_twin_whatif_maintenance.ipynb")
    _cu_cells = json.load(urllib.request.urlopen(_cu_url))["cells"]
    _cu_ip, _cu_err = get_ipython(), None
    with capture_output():
        for _cu_c in _cu_cells:
            if _cu_c.get("id") == "sec-4":
                break
            _cu_meta = _cu_c.get("metadata", {})
            if _cu_c["cell_type"] == "code" and _cu_meta.get("catchup", True):
                try:
                    _cu_src = _cu_ip.transform_cell("".join(_cu_c["source"]))
                    exec(_cu_src, _cu_ip.user_ns)
                except Exception as _cu_e:
                    _cu_err = _cu_e
                    break
    plt.close("all")
    if _cu_err is None:
        print("The earlier sections have run. Continue with the next cell.")
    else:
        print(f"The catch-up stopped with an error ({_cu_err}). "
              "Run the notebook from the top instead.")

**Why Python here.** Three years of tests do not fix a and b exactly. Python draws 2000 pairs of parameters from their uncertainty, computes an end of life for each and turns them into a distribution. It then plans the order of new battery modules, which take six months to arrive, for three futures of the battery.

**From the formulas to the code.** The parameter pairs are $(a, b) = (\hat{a}, \hat{b}) + L\,z$, with the Cholesky factor $L$ of the covariance and two standard normal numbers $z$: `futures(per_day)`. The latest order month is the largest $M$ with $P(t_{end} > t_{now} + (M + 6)/12) \ge 0.95$: `latest_order(per_day)`.

In [ ]:
L11 = math.sqrt(COV[0, 0])
L21 = COV[1, 0] / L11
L22 = math.sqrt(COV[1, 1] - L21 ** 2)                    # Cholesky factor of the 2 by 2 covariance

def futures(per_day, n=2000, seed=77):
    """End of life in years for n parameter pairs drawn from their uncertainty, with the random numbers of the lab."""
    uniform, normal = lab_rng(seed)
    out = np.zeros(n)
    for i in range(n):
        z1, z2 = normal(), normal()
        out[i] = end_of_life(A + L11 * z1, B + L21 * z1 + L22 * z2, per_day)
    return out

def latest_order(per_day, lead_months=6, level=0.95):
    """The latest month after now to order modules that arrive before the end of life with the given probability."""
    ends = futures(per_day)
    best = None
    for months in range(0, 121):
        p = float(np.mean(ends > NOW + (months + lead_months) / 12))
        if p >= level:
            best = (months, round(p, 3))
    return best

when = lambda years: (pd.Timestamp("2023-11-01") + pd.DateOffset(months=int(math.floor(years * 12 + 1e-9)))).strftime("%b %Y")
for per_day in (1.0, 1.25, 1.5):
    ends = futures(per_day)
    p5, p50, p95 = np.percentile(ends, [5, 50, 95])
    months, p = latest_order(per_day)
    print(f"{per_day} cycles per day: end of life {when(p50)} (90 percent between {when(p5)} and {when(p95)}); "
          f"order within {months} months, by {when(NOW + months / 12)}, P(in time) = {p}")

**Python note.** The Cholesky factor turns two independent normal numbers into a pair with the covariance of the fit. The strong negative correlation of a and b means that a larger calendar ageing comes with a smaller cycle ageing.

**Exercise D.** Store the latest order month after now for 1.25 cycles per day in `m_d`.

In [ ]:
m_d = None   # your answer

check("Exercise D", m_d)

In [ ]:
#@title Reference solution (run this cell; open it only after trying)
m_d = latest_order(1.25)[0]

check("Exercise D", m_d)

Part C of the [interactive lab](https://utkukose.github.io/cyber-physical-systems-NB-lecture/weeks/week-13/lab.html) opens the battery container of the campus. Paste the copied rows into `MY_ORDERS`, and the code repeats each of your plans.

In [ ]:
# Paste the results of lab Part C between the triple quotes.
MY_ORDERS = """
"""
if MY_ORDERS.strip():
    runs = pd.read_csv(io.StringIO(MY_ORDERS.strip()))
    source = "your own plans in the battery container"
else:
    runs = pd.DataFrame({"run": [1, 2, 3], "model": ["linear", "ageing", "ageing"], "cycles_per_day": [1.0, 1.0, 1.5],
                         "order_month": [40, 56, 37], "eol_years": [6.44, 8.71, 7.10], "p_in_time": [None, 0.952, 0.976]})
    source = "a simulated student"
def plan(r):
    if r.model == "linear":
        return round(0.2 / c, 2), None
    ends = futures(r.cycles_per_day)
    return round(float(np.percentile(ends, 50)), 2), round(float(np.mean(ends > NOW + (r.order_month + 6) / 12)), 3)
runs[["model_eol_years", "model_p_in_time"]] = [plan(r) for r in runs.itertuples()]
print("Data:", source)
print(runs.to_string(index=False))

In [ ]:
# Figure: 200 of the possible futures of the battery and the distribution of the end of life
fig, ax = plt.subplots(1, 2, figsize=(11, 3.6), gridspec_kw={"width_ratios": [1.5, 1]})
uniform, normal = lab_rng(77)
years = np.linspace(NOW, 10, 120)
for i in range(200):
    z1, z2 = normal(), normal()
    a, b = A + L11 * z1, B + L21 * z1 + L22 * z2
    cycles = BAT.efc.iloc[-1] + 365.25 * 1.5 * (years - NOW)
    ax[0].plot(years, Q0 * (1 - a * np.sqrt(years) - b * cycles), color="#1E6FA8", alpha=0.06, lw=1)
ax[0].plot(BAT.years, BAT.capacity_kwh, "o", ms=2.5, color="#1B2430")
ax[0].axhline(400, color="#B53F2E", ls=":", lw=1)
ax[0].set_xlabel("years since the commissioning")
ax[0].set_ylabel("capacity (kWh)")
ax[0].set_ylim(380, 505)
ax[0].set_title("1.5 cycles per day from now on", fontsize=9)
for per_day, col in ((1.0, "#2E8B57"), (1.5, "#1E6FA8")):
    ax[1].hist(futures(per_day), bins=40, color=col, alpha=0.5, label=f"{per_day} cycles per day")
months, p = latest_order(1.5)
ax[1].axvline(NOW + (months + 6) / 12, color="#B53F2E", ls="--", lw=1, label="arrival of an order in time")
ax[1].set_xlabel("end of life (years since the commissioning)")
ax[1].set_ylabel("futures")
ax[1].legend(fontsize=7.6)
plt.tight_layout()
plt.show()

**Observe and reflect.** Frequency regulation would earn money with half a cycle more per day. Which numbers would you take to the finance office to weigh that income against the earlier replacement?

## 5. The campus microgrid, stage 13

*Lecture: [The campus microgrid, stage 13: Asking the twin](https://utkukose.github.io/cyber-physical-systems-NB-lecture/weeks/week-13/lecture.html#the-campus-microgrid-stage-13)*

**Started here?** If you opened the Colab notebook at this section, run the next cell once. It runs the code of the earlier sections for you, without their printouts and figures. This takes from a few seconds to a minute, and the cell does nothing when they have already run. The Colab menu Runtime, Run before, which runs every cell above the selected one, works as well.

In [ ]:
#@title Catch-up: run this cell once if you start the notebook here
if "ANSWERS" in globals() and "latest_order" in globals():
    print("The earlier sections have already run. Continue with the next cell.")
else:
    import json, urllib.request
    import matplotlib.pyplot as plt
    from IPython import get_ipython
    from IPython.utils.capture import capture_output
    _cu_url = ("https://raw.githubusercontent.com/utkukose/"
               "cyber-physical-systems-NB-lecture/main/weeks/week-13/"
               "NB13_digital_twin_whatif_maintenance.ipynb")
    _cu_cells = json.load(urllib.request.urlopen(_cu_url))["cells"]
    _cu_ip, _cu_err = get_ipython(), None
    with capture_output():
        for _cu_c in _cu_cells:
            if _cu_c.get("id") == "sec-5":
                break
            _cu_meta = _cu_c.get("metadata", {})
            if _cu_c["cell_type"] == "code" and _cu_meta.get("catchup", True):
                try:
                    _cu_src = _cu_ip.transform_cell("".join(_cu_c["source"]))
                    exec(_cu_src, _cu_ip.user_ns)
                except Exception as _cu_e:
                    _cu_err = _cu_e
                    break
    plt.close("all")
    if _cu_err is None:
        print("The earlier sections have run. Continue with the next cell.")
    else:
        print(f"The catch-up stopped with an error ({_cu_err}). "
              "Run the notebook from the top instead.")

**Why Python here.** Stage 13 asks the twins of the campus what-if questions. The twin of the library compares heating schedules over the observed week and over 300 possible weeks. The twin of the battery predicts its end of life and plans the order of new modules. The numbers come from Sections 1 to 4.

In [ ]:
best_mc = MC[(13, "optimal")]
STAGE13 = {
    "current schedule: energy of the observed week (kWh)": now["energy"],
    "current schedule: comfort deficit of the observed week (K h)": now["deficit"],
    "13 C and optimal start: energy of the observed week (kWh)": grid[(13, "optimal")][0],
    "13 C and optimal start: comfort deficit of the observed week (K h)": grid[(13, "optimal")][1],
    "300 weeks, current schedule: P(deficit > 1 K h)": MC[(17, 6)]["p_deficit"],
    "300 weeks, 13 C and optimal start: P(deficit > 1 K h)": best_mc["p_deficit"],
    "300 weeks, 13 C and optimal start: mean energy (kWh)": best_mc["mean_kwh"],
    "300 weeks, 19 C at night: mean energy (kWh)": MC[(19, 6)]["mean_kwh"],
    "battery: end of life, linear model (years)": round(0.2 / c, 2),
    "battery: end of life, calendar and cycle model (years)": round(end_of_life(A, B, 1.0), 2),
    "battery: latest order (months from now)": latest_order(1.0)[0],
    "frequency regulation: end of life (years)": round(float(np.percentile(futures(1.5), 50)), 2),
    "frequency regulation: latest order (months from now)": latest_order(1.5)[0],
}
for key, value in STAGE13.items():
    print(f"{key:<68} {value}")

**Python note.** Every value of `STAGE13` is computed again from the functions of the earlier sections, so the table changes with them if a section is changed.

In [ ]:
# Figure: the trade-off of the schedules over 300 weeks and the order dates of the battery
fig, ax = plt.subplots(1, 2, figsize=(11, 3.4))
labels = {(17, 6): "17 C, 6:00", (19, 6): "19 C, 6:00", (17, "optimal"): "17 C, optimal",
          (15, "optimal"): "15 C, optimal", (13, "optimal"): "13 C, optimal"}
for key, m in MC.items():
    ax[0].plot(m["mean_kwh"], m["p_deficit"], "o", ms=7, color="#1E6FA8" if key[1] == "optimal" else "#C98510")
    ax[0].annotate(labels[key], (m["mean_kwh"], m["p_deficit"]), xytext=(5, 4), textcoords="offset points", fontsize=8)
ax[0].set_xlabel("mean heating energy of a week (kWh)")
ax[0].set_ylabel("P(comfort deficit > 1 K h)")
ax[0].set_ylim(-0.05, 1.1)
duties = [1.0, 1.25, 1.5]
orders = [latest_order(n)[0] for n in duties]
ax[1].bar([str(n) for n in duties], orders, color=["#2E8B57", "#C98510", "#1E6FA8"])
for k, m in enumerate(orders):
    ax[1].text(k, m + 1, f"{m} months", ha="center", fontsize=8)
ax[1].set_xlabel("cycles per day from now on")
ax[1].set_ylabel("latest order (months from now)")
ax[1].set_ylim(0, 70)
plt.tight_layout()
plt.show()

**Exercise E.** Store the latest order month for the battery with frequency regulation, as kept in `STAGE13`, in `m_e`.

In [ ]:
m_e = None   # your answer

check("Exercise E", m_e)

In [ ]:
#@title Reference solution (run this cell; open it only after trying)
m_e = STAGE13["frequency regulation: latest order (months from now)"]

check("Exercise E", m_e)

**Observe and reflect.** Which number of stage 13 would you put first in a report to the board of the university, and why?

## 6. Your application: The application switch

*Lecture: [Your application](https://utkukose.github.io/cyber-physical-systems-NB-lecture/weeks/week-13/lecture.html#your-application)*

**Started here?** If you opened the Colab notebook at this section, run the next cell once. It runs the code of the earlier sections for you, without their printouts and figures. This takes from a few seconds to a minute, and the cell does nothing when they have already run. The Colab menu Runtime, Run before, which runs every cell above the selected one, works as well.

In [ ]:
#@title Catch-up: run this cell once if you start the notebook here
if "ANSWERS" in globals() and "STAGE13" in globals():
    print("The earlier sections have already run. Continue with the next cell.")
else:
    import json, urllib.request
    import matplotlib.pyplot as plt
    from IPython import get_ipython
    from IPython.utils.capture import capture_output
    _cu_url = ("https://raw.githubusercontent.com/utkukose/"
               "cyber-physical-systems-NB-lecture/main/weeks/week-13/"
               "NB13_digital_twin_whatif_maintenance.ipynb")
    _cu_cells = json.load(urllib.request.urlopen(_cu_url))["cells"]
    _cu_ip, _cu_err = get_ipython(), None
    with capture_output():
        for _cu_c in _cu_cells:
            if _cu_c.get("id") == "sec-6":
                break
            _cu_meta = _cu_c.get("metadata", {})
            if _cu_c["cell_type"] == "code" and _cu_meta.get("catchup", True):
                try:
                    _cu_src = _cu_ip.transform_cell("".join(_cu_c["source"]))
                    exec(_cu_src, _cu_ip.user_ns)
                except Exception as _cu_e:
                    _cu_err = _cu_e
                    break
    plt.close("all")
    if _cu_err is None:
        print("The earlier sections have run. Continue with the next cell.")
    else:
        print(f"The catch-up stopped with an error ({_cu_err}). "
              "Run the notebook from the top instead.")

**Why Python here.** Wherever a condition degrades step by step, the same steps predict the remaining useful life. Fit a model of the degradation, draw its parameters from their uncertainty and find when each future crosses the limit. The function of this section does this for a system of your choice.

The systems, their models and their limits are illustrative. Set `APP` to one of the six names and run the cell.

In [ ]:
APP = "pump_bearing"
# choose: pump_bearing, led_streetlights, pv_modules, hvac_filter, tram_brake_pads, server_fans
APPS = {  # description, unit, months observed, model, start value, rate per month, noise, limit
    "pump_bearing": ("the vibration of a cooling water pump", "mm/s", 24, "exponential", 1.2, 0.045, 0.06, 7.1),
    "led_streetlights": ("the light output of the street lights of the campus", "percent", 36, "linear", 100.0, -0.35,
                         0.6, 70.0),
    "pv_modules": ("the performance ratio of the solar plant", "percent", 48, "linear", 84.0, -0.05, 0.4, 76.0),
    "hvac_filter": ("the pressure drop over the air filter of the library", "Pa", 8, "exponential", 80.0, 0.09, 0.03,
                    250.0),
    "tram_brake_pads": ("the thickness of the brake pads of a tram", "mm", 10, "linear", 18.0, -0.9, 0.15, 3.0),
    "server_fans": ("the speed of the fans of a server as their bearings wear", "rpm", 30, "exponential", 2400.0,
                    0.012, 0.01, 4200.0),
}

def app_rul(app, seed=5, draws=2000):
    desc, unit, months, model, start, rate, noise, limit = APPS[app]
    rng = np.random.default_rng(seed)
    t = np.arange(1, months + 1, dtype=float)
    if model == "linear":
        y = start + rate * t + noise * rng.normal(size=months)
        z = y                                                  # fit y = p0 + p1 t
    else:
        y = start * np.exp(rate * t) * np.exp(noise * rng.normal(size=months))
        z = np.log(y)                                          # fit log y = p0 + p1 t
    X = np.column_stack([np.ones(months), t])
    p, *_ = np.linalg.lstsq(X, z, rcond=None)
    resid = z - X @ p
    cov = resid @ resid / (months - 2) * np.linalg.inv(X.T @ X)
    target = limit if model == "linear" else math.log(limit)
    pars = rng.multivariate_normal(p, cov, size=draws)
    cross = (target - pars[:, 0]) / pars[:, 1]                 # month at which each future reaches the limit
    rul = cross - months
    print(f"{app}: {desc}, limit {limit} {unit}")
    print(f"  {months} months observed, last value {y[-1]:.2f} {unit}")
    print(f"  remaining useful life: median {np.median(rul):.1f} months, 90 percent between {np.percentile(rul, 5):.1f} "
          f"and {np.percentile(rul, 95):.1f} months")
    print(f"  plan the maintenance within {np.percentile(rul, 5):.1f} months to be in time with a probability of 95 percent")

app_rul(APP)

**Python note.** The function `app_rul` fits a straight line to the values or to their logarithm, so an exponential growth becomes a straight line too. The method `multivariate_normal` draws the parameter pairs with their covariance, like the Cholesky factor of Section 4.

**Observe and reflect.** For which system is the remaining useful life most uncertain, and what would make it more certain?

## Application challenges

Each challenge transfers the ideas of the week to a field of computing. Choose the one closest to your interests and solve it in a copy of this Colab notebook.

| Field | Challenge |
|---|---|
| Software engineering | Turn `schedule_twin` into a class with one method per control rule, and add a new rule without changing the existing code. |
| Embedded systems | The optimal start runs in the thermostat itself. Write it with integers only, with temperatures in tenths of a degree, and compare its start times with Section 1. |
| Internet of Things | The weather service sends a forecast of the outdoor temperature. Use the forecast instead of the current value in the optimal start, and measure what it changes. |
| Computer networks | The capacity tests arrive from the battery container over a link that loses one message in ten. How much later does the fit become as certain as with all tests? |
| Control engineering | Replace the on and off thermostat by a proportional controller of the heating power and compare the energy and the comfort over the 300 weeks. |
| Data science | Replace the normal distribution of the parameters by a bootstrap of the residuals and compare the distribution of the end of life. |
| Cybersecurity | An attacker raises the reported capacity by 5 kWh in every test of the last year. How much later would the campus order the modules, and how could the twin notice the attack? |
| Artificial intelligence | Train a small model on the first 24 months to predict the capacity of the last 12, and compare it with the calendar and cycle model. |

## After the Colab notebook

The self-assessment and the reflection are in the [interactive lab](https://utkukose.github.io/cyber-physical-systems-NB-lecture/weeks/week-13/lab.html), tabs *Check yourself* and *Reflect and export*. The weekly task and the research assignment are on the [week overview](https://github.com/utkukose/cyber-physical-systems-NB-lecture/tree/main/weeks/week-13).

---

<div align="center">

**Prof. Dr. Utku Kose**

Full Professor, Department of Computer Engineering, Süleyman Demirel University, Isparta, Türkiye  
Founding Director, AI Application and Research Center (YAZEM), Süleyman Demirel University  
Head of the Computer Science Division, Department of Computer Engineering, Süleyman Demirel University  
Additional affiliations: University of North Dakota (USA), Universidad Panamericana (Mexico City, Mexico), Vel Tech University (Chennai, India)  
IEEE Senior Member, ACM Professional Member

[ORCID 0000-0002-9652-6415](https://orcid.org/0000-0002-9652-6415) | [utkukose.com](https://www.utkukose.com) | [github.com/utkukose](https://github.com/utkukose)

[utkukose@sdu.edu.tr](mailto:utkukose@sdu.edu.tr) | [utku.kose@und.edu](mailto:utku.kose@und.edu) | [ukose@up.edu.mx](mailto:ukose@up.edu.mx) | [utkukose@gmail.com](mailto:utkukose@gmail.com)

This course is updated in line with current developments in the field. First version: November 2026.

</div>